# Layer Pruning (TrimLLM)

Drop the last ~25% of transformer layers, recover with QLoRA + LISA.

**Papers**
- TrimLLM https://arxiv.org/abs/2412.11242
- Reassessing Layer Pruning https://arxiv.org/abs/2411.15558
- LISA https://arxiv.org/abs/2403.17919

In [ ]:
# ###### Colab bootstrap ######
# On Colab the [experiments] extras pulls both requirements_package.txt and
# requirements_experiments.txt in one pip command — single source of truth lives
# in setup.py's extras_require. Then bootstrap() loads Colab Secrets into
# os.environ and brings up Tailscale so *.ts.net hostnames are reachable.
# Locally bootstrap() just loads .env and checks the tailnet — no installs.
#
# Required Colab Secrets (key icon → Add new secret → toggle "Notebook access"):
#   TAILSCALE_AUTHKEY   – from https://login.tailscale.com/admin/settings/keys
#   MLFLOW_TRACKING_URI, MLFLOW_EXPERIMENT_NAME, MLFLOW_TRACKING_INSECURE_TLS
#   MINIO_ENDPOINT / MINIO_ACCESS_KEY / MINIO_SECRET_KEY / MINIO_SECURE
#   HF_TOKEN
import os, subprocess, sys
IN_COLAB = "COLAB_GPU" in os.environ or "google.colab" in sys.modules
if IN_COLAB:
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "burnit_bg[experiments] @ git+https://github.com/kirilyotov/BurnIT-BG.git",
    ])
    # Colab ships a broken wandb 0.x where `wandb.proto.wandb_telemetry_pb2`
    # is missing the `Imports` symbol, which trl imports unconditionally at
    # `from trl import SFTTrainer`. We don't use wandb (report_to=[]), so the
    # cleanest fix is to uninstall it before trl is imported anywhere.
    subprocess.call([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "wandb"])
    # PEFT 0.17+ requires torchao >= 0.16.0 for is_torchao_available() checks
    # during non-4bit LoRA setup. Colab ships 0.10.0 → ImportError at
    # apply_lora() time. Upgrade here, before any PEFT code runs.
    subprocess.call([sys.executable, "-m", "pip", "install", "-q", "--upgrade", "torchao"])
try:
    import unsloth  # noqa: F401
except Exception:
    pass

from utils.colab import bootstrap
bootstrap()


In [ ]:
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"Device: {props.name}")
    print(f"VRAM:   {props.total_memory / 1024**3:.2f} GB")
    print(f"Compute capability: {props.major}.{props.minor}")


## 1. Setup & config

In [ ]:
import sys, os
from pathlib import Path

REPO_ROOT = Path.cwd()
while REPO_ROOT != REPO_ROOT.parent and not (REPO_ROOT / 'data_platform').exists():
    REPO_ROOT = REPO_ROOT.parent
sys.path.insert(0, str(REPO_ROOT))

from data_platform.common import set_env
from data_platform.storage import MinioStorage

from experiments.shared.mlflow_utils import setup_run, log_responses, stage, log_training_curves
from experiments.shared.judge_utils import judge_and_log, JudgePanel
from experiments.shared.mlflow_judges import get_scorers, batteries_to_eval_dataset, reset_cache as reset_judge_cache
from experiments.shared.prompt_registry import register_burnit_prompts
from experiments.shared.inference_utils import (
    run_full_test_battery,
    TEST_PROMPTS_IN_DOMAIN, TEST_PROMPTS_OUT_OF_DOMAIN, TEST_PROMPTS_EDGE,
    format_prompt,
)
from experiments.shared.model_utils import (
    DEFAULT_MODEL_NAME, load_model_unsloth, apply_qlora, apply_dora, apply_lora,
    count_trainable_params, cuda_device_info, default_repo_for, push_to_hf,
)
from experiments.shared.eval_utils import (
    compute_perplexity, benchmark_speed, vram_snapshot,
    compute_bertscore, compute_rouge,
    plot_train_eval_loss, plot_grad_norm, plot_metric_scorecard, overfit_summary,
    per_epoch_metrics_table, plot_per_epoch_metrics,
)
from experiments.shared.notebook_utils import log_executed_notebook
from experiments.shared.dataset_utils import (
    load_alpaca_dataset, dataset_statistics, alpaca_to_prompt,
)
from experiments.shared.dataset_browser import list_datasets, pick_dataset, download_dataset, resolve
from experiments.shared.dataset_loader import get_dataset, LoadedDataset
from transformers import EarlyStoppingCallback


In [ ]:
# ===== TOP-OF-NOTEBOOK CONFIG (override via env or edit here) =====
# Recovery fine-tuning runs after pruning ~25% of decoder blocks. We default
# to the same Llama-3.2-1B + 2-dataset slice as the QLoRA baseline so the
# pruned model can be compared apples-to-apples against the unpruned run.
# Swap the recovery method by editing cell 14: apply_qlora can be replaced
# with apply_dora / apply_lora (for bf16 base) / GaLore-style full FT.

import torch
from experiments.shared.datasets_registry import Datasets, MENTAL_HEALTH_ALL

MENTALHEALTH: tuple[str, ...] = (
    Datasets.CHITANKA_BG.value.id,
    Datasets.MH_COUNSELING_BG.value.id,
)

MODEL_NAME          = os.getenv("MODEL_NAME",       'meta-llama/Llama-3.2-1B-Instruct')
DATASET_NAME        = os.getenv("DATASET_NAME")   or ",".join(MENTALHEALTH)
DATASET_SOURCE      = os.getenv("DATASET_SOURCE",   'registry')
DATASET_SUBSET      = os.getenv("DATASET_SUBSET",   None)
TRAIN_EPOCHS        = int(os.getenv("TRAIN_EPOCHS", "1"))
EARLY_STOP_PATIENCE = int(os.getenv("EARLY_STOP_PATIENCE", "1"))
_max_steps_env      = os.getenv("MAX_STEPS")
MAX_STEPS           = int(_max_steps_env) if _max_steps_env else None
METRIC_FOR_BEST     = os.getenv("METRIC_FOR_BEST_MODEL", 'eval_loss')

# Pruning-specific knobs
PRUNE_RATIO         = float(os.getenv("PRUNE_RATIO", "0.25"))   # drop trailing 25%
BASELINE_RUN_ID     = os.getenv("BASELINE_RUN_ID")              # MLflow run to load weights from

print(f"[config] model={MODEL_NAME}  dataset={DATASET_NAME} ({DATASET_SOURCE})  "
      f"epochs={TRAIN_EPOCHS}  early_stop_patience={EARLY_STOP_PATIENCE}  "
      f"prune_ratio={PRUNE_RATIO}")

# Mixed-precision auto-detect.
# torch.cuda.is_bf16_supported() returns True on T4 (Turing 7.5) where bf16
# *training* actually fails — SFTConfig/TrainingArguments reject bf16=True on
# non-Ampere GPUs. Compute capability >= 8.0 is the only reliable signal.
def _has_bf16_training():
    if not torch.cuda.is_available():
        return False
    major, _ = torch.cuda.get_device_capability(0)
    return major >= 8
USE_BF16 = _has_bf16_training()
USE_FP16 = bool(torch.cuda.is_available() and not USE_BF16)
# Thread this through load_model_unsloth(..., dtype=MODEL_DTYPE) so the base
# model and the trainer's AMP path agree on dtype. Mismatch triggers
# "_amp_foreach_non_finite_check_and_unscale_cuda not implemented for BFloat16".
MODEL_DTYPE = torch.bfloat16 if USE_BF16 else (torch.float16 if USE_FP16 else torch.float32)
print(f"[config] USE_BF16={USE_BF16} USE_FP16={USE_FP16} MODEL_DTYPE={MODEL_DTYPE}")

# Per-notebook identity tags. METHOD is constant for this notebook; MODEL_TAG
# is the sanitised basename of MODEL_NAME; MLFLOW_EXPERIMENT groups runs by
# method+model so each combination has its own MLflow experiment.
METHOD = "layer-pruning"
MODEL_TAG = MODEL_NAME.split("/")[-1].lower().replace("_", "-")
MLFLOW_EXPERIMENT = f"{METHOD}_{MODEL_TAG}"
print(f"[config] METHOD={METHOD}  MODEL_TAG={MODEL_TAG}  "
      f"MLFLOW_EXPERIMENT={MLFLOW_EXPERIMENT}")

# HuggingFace branch / repo identity. Convention used across all BurnIT-BG
# experiments: ONE model repo per project, ONE branch per (experiment_type,
# algorithm, model, dataset) combination — so apples-to-apples comparison
# across iterations stays clean.
from experiments.shared.model_utils import hf_branch_name

EXPERIMENT_TYPE = "pruning"
HF_REPO = os.getenv("HF_MODEL_REPO", "kiplayo/BurnIT-BG")
HF_BRANCH = os.getenv(
    "HF_REVISION",
    hf_branch_name(EXPERIMENT_TYPE, METHOD, MODEL_NAME, DATASET_NAME),
)
print(f"[hf] repo={HF_REPO}  branch={HF_BRANCH}")


In [ ]:
# MLflow needs to be in this cell's scope: mlflow.genai.evaluate (below)
# and the model-save block both reference it directly.
import mlflow
set_env(quiet=True)
# Throttle mlflow.genai.evaluate's scorer thread pool BEFORE the panel runs.
# Default is 4 parallel scorers; with our 3-call panel that'd be 12 concurrent
# requests to a 40 RPM NVIDIA endpoint — instant 429 storm. We use 2 workers
# and let the per-model RpmBucket in NvidiaChatClient pace the wire.
os.environ.setdefault("MLFLOW_GENAI_EVAL_MAX_SCORER_WORKERS", "2")
# Bump the urllib3 connection pool so concurrent MLflow log_metric / log_artifact
# calls during mlflow.genai.evaluate don't trip the "Connection pool is full,
# discarding connection" warning (default pool size is 10 — too small for the
# evaluator's burst of writes back to the MLflow tracking server).
try:
    import urllib3 as _u3
    _u3.PoolManager(num_pools=10, maxsize=50)  # warms up larger pools
    import requests.adapters as _ra
    _ra.DEFAULT_POOLSIZE = 50
except Exception:
    pass
# Optional per-process RPM ceiling (per NVIDIA model key). Free tier is ~40
# RPM per key; we default to 30 to leave headroom for retries. Override via
# the NVIDIA_DEFAULT_RPM env var in Colab Secrets if you have a higher quota.
os.environ.setdefault("NVIDIA_DEFAULT_RPM", "30")

tracking, tags, run_name = setup_run(
    experiment='layer_pruning',
    model=MODEL_NAME,    # was DEFAULT_MODEL_NAME — show the actually-used model in MLflow tags
    stage="experiment",
    mlflow_experiment_name=MLFLOW_EXPERIMENT,
)
print(f"run_name = {run_name}")
print(f"tags     = {tags}")
print(f"machine  = {cuda_device_info()}")

# ─── Pre-run logging buffer ────────────────────────────────────────────────
# Cells 10/12 measure layer similarity / prune stats BEFORE cell 14 opens the
# training run via `with tracking.run(...)`. Direct tracking.log_* calls would
# fail with "An active MLflow run is required". Buffer here, flush at run open.
_PRE_RUN_METRICS: dict = {}
_PRE_RUN_PARAMS:  dict = {}

def _log_metrics_safely(d: dict) -> None:
    """Log metrics if a run is active, otherwise stash for later flush."""
    import mlflow as _ml
    if _ml.active_run():
        tracking.log_metrics(d)
    else:
        _PRE_RUN_METRICS.update(d)

def _log_params_safely(d: dict) -> None:
    """Log params if a run is active, otherwise stash for later flush."""
    import mlflow as _ml
    if _ml.active_run():
        tracking.log_params(d)
    else:
        _PRE_RUN_PARAMS.update(d)

def _flush_pre_run_logs() -> None:
    """Called at the top of `with tracking.run(...)` in cell 14."""
    if _PRE_RUN_PARAMS:
        tracking.log_params(_PRE_RUN_PARAMS)
        print(f"[pre-run] flushed {len(_PRE_RUN_PARAMS)} buffered params")
    if _PRE_RUN_METRICS:
        tracking.log_metrics(_PRE_RUN_METRICS)
        print(f"[pre-run] flushed {len(_PRE_RUN_METRICS)} buffered metrics")


## 2. Data loading

In [ ]:
# ###### Load the dataset configured at the top + register in MLflow ######
import mlflow as _mlflow

_ds = get_dataset(DATASET_NAME, source=DATASET_SOURCE, subset=DATASET_SUBSET)
train_records, eval_records = _ds.train, _ds.eval
train_stats = dataset_statistics(train_records)
eval_stats  = dataset_statistics(eval_records)
print(f"train: {len(train_records)} rows  eval: {len(eval_records)} rows  "
      f"({_ds.source}: {_ds.source_uri})")
print(train_stats)

# Register the dataset against the active run so it appears in MLflow's
# Datasets tab. If no run is active yet (the typical case here — training
# opens its own run below), we'll re-attach there.
try:
    if _mlflow.active_run():
        _mlflow.log_input(_ds.to_mlflow_input(context='train'), context='train')
        _mlflow.log_input(_ds.to_mlflow_input(context='eval'), context='eval')
        print(f"[mlflow] dataset registered: {_ds.name}")
except Exception as _ds_exc:
    print(f"[mlflow] dataset register skipped: {_ds_exc}")


## 2b. Load the baseline + measure layer similarity

In [ ]:
# Load baseline weights from MLflow (replace with your baseline run-id).
# Falls back to the bare base model if no baseline run is given.
HF_TOKEN = os.getenv("HF_TOKEN")
MAX_SEQ_LEN = 1024

model, tokenizer = load_model_unsloth(
    MODEL_NAME,
    max_seq_length=MAX_SEQ_LEN,
    load_in_4bit=True,
    token=HF_TOKEN,
    dtype=MODEL_DTYPE,
)
if BASELINE_RUN_ID:
    try:
        local_path = tracking.load_model(run_id=BASELINE_RUN_ID, artifact_path="model")
        print(f"loaded baseline weights from {local_path}")
    except Exception as exc:
        print(f"[warn] baseline load failed ({exc}); continuing with bare base model")

# Layer similarity: capture hidden states with forward hooks, compute
# cosine similarity between consecutive layers.
import torch
from torch.nn.functional import cosine_similarity

@torch.no_grad()
def measure_layer_similarity(model, tokenizer, samples, max_tokens=256):
    layers = model.model.layers if hasattr(model, "model") else model.layers
    hidden_per_layer = []
    handles = []
    def make_hook(idx):
        def _hook(_module, _input, output):
            h = output[0] if isinstance(output, tuple) else output
            hidden_per_layer.append((idx, h.mean(dim=1).detach().cpu()))
        return _hook
    for i, layer in enumerate(layers):
        handles.append(layer.register_forward_hook(make_hook(i)))
    try:
        for text in samples:
            ids = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_tokens).to(model.device)
            model(**ids)
    finally:
        for h in handles: h.remove()
    # Aggregate per layer
    by_layer = {}
    for i, h in hidden_per_layer:
        by_layer.setdefault(i, []).append(h)
    means = [torch.cat(by_layer[i]).mean(dim=0) for i in sorted(by_layer)]
    sims = []
    for i in range(len(means) - 1):
        a, b = means[i], means[i+1]
        sims.append(float(cosine_similarity(a, b, dim=0).item()))
    return sims

similarities = measure_layer_similarity(
    model, tokenizer,
    samples=[alpaca_to_prompt(r) for r in eval_records[:32]],
)
import matplotlib.pyplot as plt
plt.figure(figsize=(9, 3.5))
plt.plot(range(len(similarities)), similarities, marker="o")
plt.axhline(0.95, color="crimson", linestyle="--", label="prune-candidate threshold")
plt.xlabel("Layer index"); plt.ylabel("cos(layer_i, layer_{i+1})")
plt.title("Layer-to-layer similarity"); plt.legend(); plt.tight_layout()
plt.savefig("/tmp/layer_sim.png", dpi=140)
_log_metrics_safely({f"layer_sim.{i}": s for i, s in enumerate(similarities)})
plt.show()


## 3. Prune the last 25% of layers

In [ ]:
# Remove the trailing ~25% of decoder blocks.
total_layers = len(model.model.layers)
prune_from = int(round(total_layers * 0.75))
keep = model.model.layers[:prune_from]
removed = total_layers - len(keep)
print(f"pruning layers {prune_from}..{total_layers - 1} ({removed} layers)")

import torch.nn as nn
model.model.layers = nn.ModuleList(keep)
if hasattr(model.config, "num_hidden_layers"):
    model.config.num_hidden_layers = len(keep)

params_after = count_trainable_params(model)
_log_params_safely({
    "prune.total_layers_before": total_layers,
    "prune.total_layers_after": len(keep),
    "prune.removed_layers": removed,
    "prune.params_after": params_after["total"],
})


## 4. Recovery fine-tuning (QLoRA + LISA)

After dropping ~25 % of layers the model loses capability. We rebuild it with:

| Component | What it does | Why now |
|---|---|---|
| **QLoRA** | Adds small trainable LoRA adapters on the 4-bit base | Fast, low VRAM, recovers most of the lost capacity in 1 epoch |
| **LISA** | Random subset of layers trainable per step (rest frozen) | Targets different layers each step — well-suited to pruned models because no single survivor needs to absorb everything at once |

**Alternative recovery algorithms** — swap the `apply_qlora(...)` line in the cell below:

| Recovery method | Quality vs. QLoRA | Wall-clock vs. QLoRA |
|---|---|---|
| `apply_dora(model, r=8, lora_alpha=16)` | +1-3 pts | ~+15 % |
| `apply_lora(model, r=8, lora_alpha=16)` (use full-precision base) | comparable, less quant noise | ~+10 % (more VRAM) |
| GaLore full FT (`optim="galore_adamw_8bit"`, no `apply_*` call) | **best** — adjusts every surviving weight | ~3-4× slower |

For a first run, **stick with QLoRA + LISA**. Once you see how much accuracy is recoverable, decide whether DoRA / GaLore is worth the extra time.


In [ ]:
# ─── Recovery fine-tuning: QLoRA + LISA on the pruned model ────────────────
# RECOVERY METHOD CHOICE
#   Default: apply_qlora — fastest, lowest VRAM. Good for first-pass recovery.
#   Alternatives (swap the apply_* line below):
#     apply_dora(model, r=8, lora_alpha=16)             # magnitude+direction; +1-3 pts at +15% cost
#     apply_lora(model, r=8, lora_alpha=16)             # plain LoRA on bf16 base (set load_in_4bit=False above)
#     (none — for GaLore full-FT recovery; replace optim="galore_adamw_8bit"
#      and skip apply_qlora entirely. Best recovery quality, slowest.)
# LISA stays on regardless of method — it works by toggling layer-level
# requires_grad each step, which is method-agnostic.

model = apply_qlora(model, r=8, lora_alpha=16, use_gradient_checkpointing=False)

# AMP safety net — Unsloth picks bf16 for LoRA adapters on any GPU reporting
# bf16 support; on L4 (Ampere+) this lines up with our MODEL_DTYPE so the
# cast is usually a no-op. Defensive — no-op if dtypes already match.
import torch as _t
_n_cast = 0
if isinstance(MODEL_DTYPE, _t.dtype):
    for _p in model.parameters():
        if _p.requires_grad and _p.dtype != MODEL_DTYPE:
            _p.data = _p.data.to(MODEL_DTYPE)
            _n_cast += 1
if _n_cast:
    print(f"[adapter_dtype] aligned {_n_cast} trainable params to {MODEL_DTYPE}")

# LISA: at each step, freeze all decoder blocks except a randomly-chosen K.
# This dramatically lowers VRAM and trains a different subset every step.
# Particularly useful after pruning — the remaining layers don't all need to
# absorb the lost capacity at once.
import random
ACTIVE_LAYERS_PER_STEP = 4

# Robust decoder-layer accessor. Unsloth's fast-path wrapping has
# ``model.model.layers``; the plain PEFT fallback adds one more wrapper so
# the layers live at ``model.base_model.model.model.layers`` (or, via
# PEFT's ``__getattr__`` delegation, at ``model.model.model.layers``). Try
# the known paths in order and return the first one that resolves to a
# ModuleList-like.
def _decoder_layers(m):
    import torch.nn as _nn
    candidates = [
        # Unsloth fast path / direct LlamaModel
        ("model", "layers"),
        # Plain PEFT delegation: peft.model → LlamaForCausalLM, +.model → LlamaModel
        ("model", "model", "layers"),
        # Explicit PEFT chain (some peft versions)
        ("base_model", "model", "model", "layers"),
        ("base_model", "model", "layers"),
        # Bare LlamaModel (no wrapper)
        ("layers",),
    ]
    for path in candidates:
        try:
            obj = m
            for attr in path:
                obj = getattr(obj, attr)
            # Sanity check — should be iterable of nn.Module
            if hasattr(obj, "__len__") and len(obj) > 0 and isinstance(obj[0], _nn.Module):
                return obj
        except AttributeError:
            continue
    raise AttributeError(
        f"Could not find decoder layers on {type(m).__name__}; "
        f"checked paths: {candidates}"
    )


def lisa_select(model, k=ACTIVE_LAYERS_PER_STEP):
    layers = _decoder_layers(model)
    chosen = set(random.sample(range(len(layers)), k=min(k, len(layers))))
    for i, layer in enumerate(layers):
        for p in layer.parameters():
            p.requires_grad_(i in chosen)
    return chosen

from datasets import Dataset
from trl import SFTTrainer, SFTConfig
from transformers import TrainerCallback

train_ds = Dataset.from_list(train_records).map(
    lambda r: {**r, "text": alpaca_to_prompt(r, eos_token=tokenizer.eos_token)})
eval_ds  = Dataset.from_list(eval_records).map(
    lambda r: {**r, "text": alpaca_to_prompt(r, eos_token=tokenizer.eos_token)})

class LISACallback(TrainerCallback):
    def on_step_begin(self, args, state, control, **kw):
        lisa_select(kw["model"])

OUTPUT_DIR = REPO_ROOT / "tmp/experiments/layer_pruning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
training_args = SFTConfig(
    output_dir=str(OUTPUT_DIR),
    num_train_epochs=TRAIN_EPOCHS,
    per_device_train_batch_size=16,    # L4 24GB fits batch 16 at MAX_SEQ_LEN=1024
    gradient_accumulation_steps=1,
    learning_rate=1e-4,                # gentler than baseline 2e-4 — recovery, not fresh train
    warmup_ratio=0.03,
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    bf16=True, fp16=False,             # L4 native bf16, no GradScaler
    optim="adamw_8bit",
    report_to=[],
    lr_scheduler_type="cosine",
    max_grad_norm=1.0,
    load_best_model_at_end=True,
    metric_for_best_model=METRIC_FOR_BEST,
    greater_is_better=False,
    eval_strategy="epoch",
    dataset_text_field="text",
    max_length=MAX_SEQ_LEN,
)

with tracking.run(run_name=run_name, tags=tags):
    _flush_pre_run_logs()
    tracking.log_params({
        "lisa.active_per_step": ACTIVE_LAYERS_PER_STEP,
        "prune.ratio": PRUNE_RATIO,
        "recovery.method": "qlora+lisa",
    })
    with stage(tracking, "recovery_train"):
        trainer = SFTTrainer(
            model=model,
            processing_class=tokenizer,
            train_dataset=train_ds,
            eval_dataset=eval_ds,
            args=training_args,
            callbacks=[
                EarlyStoppingCallback(early_stopping_patience=EARLY_STOP_PATIENCE),
                LISACallback(),
            ],
        )
        trainer.train()
        steps = [h["step"] for h in trainer.state.log_history if "loss" in h]
        losses = [h["loss"] for h in trainer.state.log_history if "loss" in h]
        if steps:
            log_training_curves(tracking, steps=steps, losses=losses,
                                title="layer-pruning recovery")
        # ─── per-epoch metrics table ────────────────────────────────────────
        # Same columns SFTTrainer prints live (epoch / training_loss /
        # validation_loss / entropy / mean_token_accuracy / num_tokens).
        try:
            _epoch_rows = per_epoch_metrics_table(list(trainer.state.log_history))
            if _epoch_rows:
                tracking.log_plot(
                    plot_per_epoch_metrics(_epoch_rows,
                                           title="layer-pruning — per-epoch metrics"),
                    key="layer_pruning_epoch_metrics_table",
                )
                try:
                    import mlflow as _mlflow
                    _mlflow.log_table(data=_epoch_rows,
                                      artifact_file="layer_pruning_epoch_metrics.json")
                except Exception as _exc:
                    print(f"[metrics_table] log_table skipped: {_exc}")
                print(f"[metrics_table] logged {len(_epoch_rows)} epoch rows")
            else:
                print("[metrics_table] no eval rows in trainer.state.log_history")
        except Exception as _exc:
            print(f"[metrics_table] failed: {_exc}")


## 5–7. Evaluation — perplexity & speed comparison → inference test → LLM-judge → save

In [ ]:
# ###### Evaluate + inference test + LLM-judge + save (one MLflow run) ######
# If the training cell above left a run open we reuse it; otherwise we open a
# fresh run with the same name/tags so every eval/judge/save metric is logged.
import mlflow
from contextlib import nullcontext

OUTPUT_DIR = REPO_ROOT / "tmp/experiments/layer_pruning"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
_run_ctx = nullcontext() if mlflow.active_run() else tracking.run(run_name=run_name, tags=tags)
with _run_ctx:
    # --- section 5: evaluation (folded in so it always has an active run) ---
    with stage(tracking, "evaluate"):
        ppl = compute_perplexity(model, tokenizer, [r["output"] for r in eval_records[:64]])
        speed = benchmark_speed(model, tokenizer, new_tokens=128, runs=3)
        _log_metrics_safely({
            "eval_perplexity": float(ppl),
            "tokens_per_sec": speed["tokens_per_sec_mean"],
            **{f"vram.{k}": v for k, v in vram_snapshot().items()},
        })
        print(f"perplexity={ppl:.3f}  tokens/sec={speed['tokens_per_sec_mean']:.1f}")

    # --- inference test battery ---
    with stage(tracking, "inference_test"):
        batteries = run_full_test_battery((model, tokenizer), max_new_tokens=256)
        log_responses(tracking, experiment='layer_pruning', model_checkpoint=str(OUTPUT_DIR), **batteries)
        for k, v in batteries.items():
            print(f"-- {k} --")
            for entry in v[:2]:
                print(f"  Q: {entry['prompt'][:80]}\n  A: {entry['response'][:200]}\n")

    # --- LLM-judge panel (Mistral quality + Llama-Guard/Nemotron safety) ---
    with stage(tracking, "judge"):
        # Path A — custom panel (log_table artifact + summary metrics on this run)
        judge_result = judge_and_log(tracking, experiment='layer_pruning', batteries=batteries)
        print("judge summary:", judge_result["summary"])
    with stage(tracking, "judge_mlflow_eval"):
        # Path B — MLflow GenAI Evaluation API (per-row Feedback in Evaluation tab).
        # Result cache means the panel is called ONCE per response total across A+B.
        # reset_judge_cache() removed — see judge_utils.judge_and_log; the singleton-shared cache between path A + path B is the whole point.
        eval_data = batteries_to_eval_dataset(batteries)
        try:
            mlflow.genai.evaluate(data=eval_data, scorers=get_scorers())
            print(f"mlflow.genai.evaluate -> {len(eval_data)} rows scored")
        except Exception as exc:
            print(f"[mlflow.genai.evaluate] skipped: {type(exc).__name__}: {exc}")

    # --- metric plots + scorecard ---
    with stage(tracking, "metrics_plots"):
        try:
            history = list(trainer.state.log_history) if "trainer" in dir() else []
        except Exception:
            history = []
        if history:
            try: tracking.log_plot(plot_train_eval_loss(history, title=f"layer_pruning loss"), key=f"layer_pruning_train_eval_loss")
            except Exception as exc: print(f"[plots] loss skipped: {exc}")
            try: tracking.log_plot(plot_grad_norm(history, title=f"layer_pruning grad"), key=f"layer_pruning_grad_norm")
            except Exception as exc: print(f"[plots] grad skipped: {exc}")
            try: _log_metrics_safely({f"overfit.{k}": v for k, v in overfit_summary(history).items() if isinstance(v, (int, float))})
            except Exception: pass
        try:
            summary = judge_result.get("summary", {}) if "judge_result" in dir() else {}
            sc = {}
            ppl_val = summary.get("eval_perplexity")
            if isinstance(ppl_val, (int, float)) and ppl_val > 0:
                sc["1/(1+ppl)"] = 1.0 / (1.0 + ppl_val)
            ov = summary.get("judge.overall_mean")
            if isinstance(ov, (int, float)):
                sc["judge_overall/5"] = ov / 5.0
            un = summary.get("judge.unsafe_count"); nt = summary.get("judge.n_total")
            if isinstance(un, (int, float)) and isinstance(nt, (int, float)) and nt > 0:
                sc["1-unsafe_rate"] = 1.0 - (un / nt)
            if sc:
                tracking.log_plot(plot_metric_scorecard(sc, title=f"layer_pruning scorecard"), key=f"layer_pruning_scorecard")
        except Exception as exc:
            print(f"[plots] scorecard skipped: {exc}")

    # --- save model via MLflow (artifact store backs onto MinIO) + GGUF ---
        # save_model — robust to PEFT-wrapped models. MLflow's
    # transformers flavor rejects PeftModelForCausalLM ("must be a
    # Pipeline / dict / path"). We save the model+tokenizer locally
    # via save_pretrained (works for PEFT and full models) and log the
    # directory as plain artifacts. If MLflow is reachable we also try
    # to register the resulting artifact under the configured name.
    with stage(tracking, "save_model"):
        import tempfile
        from pathlib import Path as _Path
        _save_dir = _Path(tempfile.mkdtemp(prefix="hf_model_save_"))
        try:
            model.save_pretrained(str(_save_dir))
            tokenizer.save_pretrained(str(_save_dir))
            mlflow.log_artifacts(str(_save_dir), artifact_path="model")
            print(f"[save] model + tokenizer logged as artifacts ({_save_dir})")
            try:
                _run_id = mlflow.active_run().info.run_id if mlflow.active_run() else None
                if _run_id:
                    mlflow.register_model(f"runs:/{_run_id}/model", "burnit-bg-layer-pruning")
                    print(f"[save] registered as burnit-bg-layer-pruning (version added)")
            except Exception as _reg_exc:
                print(f"[save] artifacts saved; registration skipped: {type(_reg_exc).__name__}: {_reg_exc}")
        except Exception as exc:
            print(f"[save] failed: {type(exc).__name__}: {exc}")

    # --- optional: publish to HuggingFace (set PUSH_TO_HF=1 in env/secrets) ---
    if os.getenv("PUSH_TO_HF", "0").lower() in ("1", "true", "yes"):
        with stage(tracking, "push_to_hf"):
            try:
                uri = push_to_hf(
                    model, tokenizer, HF_REPO,
                    revision=HF_BRANCH,
                    private=False,
                    experiment=METHOD,
                    method=METHOD,
                    base_model=MODEL_NAME,
                    mlflow_experiment=MLFLOW_EXPERIMENT,
                    merge_adapters=os.getenv("HF_MERGE_ADAPTERS", "0").lower() in ("1", "true", "yes"),
                )
                # Link back from MLflow to the HF page — visible on the run page
                # as a tag (clickable in UI) AND queryable as params.
                hf_url = f"https://huggingface.co/{HF_REPO}/tree/{HF_BRANCH}"
                mlflow.set_tag("hf_model_url", hf_url)
                mlflow.log_param("hf_repo", HF_REPO)
                mlflow.log_param("hf_revision", HF_BRANCH)
                mlflow.log_param("hf_uri", uri)
                print(f"[hf] pushed -> {uri}")
                print(f"[hf] browse -> {hf_url}")
            except Exception as exc:
                print(f"[hf] push skipped: {type(exc).__name__}: {exc}")

    # --- log this executed notebook (with cell outputs) to MLflow ---
    with stage(tracking, "publish_notebook"):
        try:
            log_executed_notebook(tracking, also_html=True, require_outputs=False)
        except Exception as exc:
            print(f"[notebook] publish skipped: {exc}")
    tracking.log_hardware(step=1)
